# Retail E-Commerce BI Analytics
## Data Quality & Anomaly Audit

This notebook audits the Olist source tables before KPI publication. It checks structural integrity, missingness, value domains, temporal consistency, referential integrity, and reconciliation across analytical grains.

**Important:** unusual records are not automatically deleted. Each finding must be classified as an expected characteristic, a data-quality issue, or an analytical risk.

In [21]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path('../data/raw')

orders = pd.read_csv(DATA_PATH / 'olist_orders_dataset.csv', parse_dates=[
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
])
customers = pd.read_csv(DATA_PATH / 'olist_customers_dataset.csv')
items = pd.read_csv(DATA_PATH / 'olist_order_items_dataset.csv')
products = pd.read_csv(DATA_PATH / 'olist_products_dataset.csv')
payments = pd.read_csv(DATA_PATH / 'olist_order_payments_dataset.csv')
reviews = pd.read_csv(DATA_PATH / 'olist_order_reviews_dataset.csv')

## 1. Source-table dimensions

These counts establish the expected starting point for the audit.

In [22]:
tables = {
    'orders': orders,
    'customers': customers,
    'order_items': items,
    'products': products,
    'order_payments': payments,
    'order_reviews': reviews
}

pd.DataFrame({name: {'rows': len(df), 'columns': len(df.columns)} for name, df in tables.items()}).T

,rows,columns
orders,99441,8
customers,99441,5
order_items,112650,7
products,32951,9
order_payments,103886,5
order_reviews,99224,7


## 2. Primary-key uniqueness

A duplicate key is only a defect when uniqueness is expected at that table's grain. Child tables are intentionally excluded from blanket uniqueness checks.

In [23]:
key_checks = {
    'orders.order_id': orders['order_id'].duplicated().sum(),
    'customers.customer_id': customers['customer_id'].duplicated().sum(),
    'products.product_id': products['product_id'].duplicated().sum(),
}
pd.Series(key_checks, name='duplicate_rows')

orders.order_id          0
customers.customer_id    0
products.product_id      0
Name: duplicate_rows, dtype: int64

## 3. Referential-integrity checks

Identify child records whose parent key is absent.

In [24]:
referential_checks = {
    'items.order_id_missing': (~items['order_id'].isin(orders['order_id'])).sum(),
    'items.product_id_missing': (~items['product_id'].isin(products['product_id'])).sum(),
    'orders.customer_id_missing': (~orders['customer_id'].isin(customers['customer_id'])).sum(),
    'payments.order_id_missing': (~payments['order_id'].isin(orders['order_id'])).sum(),
    'reviews.order_id_missing': (~reviews['order_id'].isin(orders['order_id'])).sum(),
}
pd.Series(referential_checks, name='orphan_rows')

items.order_id_missing        0
items.product_id_missing      0
orders.customer_id_missing    0
payments.order_id_missing     0
reviews.order_id_missing      0
Name: orphan_rows, dtype: int64

## 4. Missingness profile

Missing values are reported by table and field. Missing delivery timestamps and missing reviews can be legitimate, so they are investigated rather than automatically imputed or deleted.

In [25]:
missingness = []
for name, df in tables.items():
    for column in df.columns:
        missing = df[column].isna().sum()
        if missing:
            missingness.append({
                'table': name,
                'column': column,
                'missing_rows': missing,
                'missing_pct': round(100 * missing / len(df), 2)
            })

missingness_df = pd.DataFrame(missingness).sort_values(['table', 'missing_pct'], ascending=[True, False])
missingness_df

,table,column,missing_rows,missing_pct
11,order_reviews,review_comment_title,87656,88.34
12,order_reviews,review_comment_message,58247,58.70
2,orders,order_delivered_customer_date,2965,2.98
1,orders,order_delivered_carrier_date,1783,1.79
0,orders,order_approved_at,160,0.16
3,products,product_category_name,610,1.85
4,products,product_name_lenght,610,1.85
5,products,product_description_lenght,610,1.85
6,products,product_photos_qty,610,1.85
7,products,product_weight_g,2,0.01


## 5. Value-domain anomalies

These checks flag values outside the expected business domain. They do not decide whether every flagged row should be removed.

In [26]:
value_checks = {
    'negative_item_price': (items['price'] < 0).sum(),
    'zero_item_price': (items['price'] == 0).sum(),
    'negative_freight': (items['freight_value'] < 0).sum(),
    'negative_payment': (payments['payment_value'] < 0).sum(),
    'zero_payment': (payments['payment_value'] == 0).sum(),
    'invalid_review_score': ((reviews['review_score'] < 1) | (reviews['review_score'] > 5)).sum(),
    'negative_installments': (payments['payment_installments'] < 0).sum(),
}
pd.Series(value_checks, name='issue_count')

negative_item_price      0
zero_item_price          0
negative_freight         0
negative_payment         0
zero_payment             9
invalid_review_score     0
negative_installments    0
Name: issue_count, dtype: int64

## 6. Temporal consistency

Check whether operational timestamps violate the expected sequence.

In [27]:
temporal_checks = {
    'approval_before_purchase': ((orders['order_approved_at'].notna()) &
                                  (orders['order_approved_at'] < orders['order_purchase_timestamp'])).sum(),
    'carrier_before_purchase': ((orders['order_delivered_carrier_date'].notna()) &
                                  (orders['order_delivered_carrier_date'] < orders['order_purchase_timestamp'])).sum(),
    'delivery_before_purchase': ((orders['order_delivered_customer_date'].notna()) &
                                   (orders['order_delivered_customer_date'] < orders['order_purchase_timestamp'])).sum(),
    'delivery_before_carrier': ((orders['order_delivered_customer_date'].notna()) &
                                 (orders['order_delivered_carrier_date'].notna()) &
                                 (orders['order_delivered_customer_date'] < orders['order_delivered_carrier_date'])).sum(),
}
pd.Series(temporal_checks, name='issue_count')

approval_before_purchase      0
carrier_before_purchase     166
delivery_before_purchase      0
delivery_before_carrier      23
Name: issue_count, dtype: int64

In [31]:
timestamp_anomalies = orders[
    (
        orders["order_delivered_carrier_date"].notna()
        & (
            orders["order_delivered_carrier_date"]
            < orders["order_purchase_timestamp"]
        )
    )
    |
    (
        orders["order_delivered_customer_date"].notna()
        & orders["order_delivered_carrier_date"].notna()
        & (
            orders["order_delivered_customer_date"]
            < orders["order_delivered_carrier_date"]
        )
    )
][[
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]]

timestamp_anomalies

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
615,b9afddbdcfadc9a87b41a83271c3e888,delivered,2018-08-16 13:50:48,2018-08-16 14:05:13,2018-08-16 13:27:00,2018-08-24 14:58:37,2018-09-04
1111,ad133696906f6a78826daa0911b7daec,delivered,2018-06-15 15:41:22,2018-06-15 16:19:23,2018-06-15 14:52:00,2018-06-22 18:09:37,2018-07-18
1329,74e033208dc13a7b8127eb8e73d09b76,delivered,2018-05-02 10:48:44,2018-05-02 11:13:45,2018-05-02 09:49:00,2018-05-07 23:06:36,2018-05-29
1372,a6b58794fd2ba533359a76c08df576e3,delivered,2018-05-14 15:18:23,2018-05-14 15:33:35,2018-05-14 13:46:00,2018-05-19 19:33:32,2018-06-08
1864,5792e0b1c8c8a2bf53af468c9a422c88,delivered,2018-07-26 13:25:14,2018-07-26 13:35:14,2018-07-26 12:42:00,2018-07-30 14:45:02,2018-08-09
...,...,...,...,...,...,...,...
98172,f7780ea2807db31691e83f0013294035,delivered,2018-07-30 15:22:15,2018-07-30 15:35:16,2018-07-30 15:00:00,2018-08-02 18:32:30,2018-08-02
98430,d7646ffe8fdd9e7d9557f9f7cbf04530,delivered,2018-05-04 14:50:37,2018-05-04 15:10:22,2018-05-04 14:48:00,2018-05-08 19:06:42,2018-05-16
98672,5ded8a3706eabd813685534724f066de,delivered,2018-07-18 08:46:52,2018-07-18 09:01:48,2018-07-18 08:44:00,2018-07-25 13:53:17,2018-08-10
98780,d10046876c7d9f01613da59ffa6cb07f,delivered,2018-07-18 16:14:16,2018-07-18 16:25:17,2018-07-18 15:34:00,2018-07-23 20:46:44,2018-08-07


In [32]:
timestamp_anomalies.shape

(189, 7)

In [33]:
timestamp_anomalies.head(20)

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
615,b9afddbdcfadc9a87b41a83271c3e888,delivered,2018-08-16 13:50:48,2018-08-16 14:05:13,2018-08-16 13:27:00,2018-08-24 14:58:37,2018-09-04
1111,ad133696906f6a78826daa0911b7daec,delivered,2018-06-15 15:41:22,2018-06-15 16:19:23,2018-06-15 14:52:00,2018-06-22 18:09:37,2018-07-18
1329,74e033208dc13a7b8127eb8e73d09b76,delivered,2018-05-02 10:48:44,2018-05-02 11:13:45,2018-05-02 09:49:00,2018-05-07 23:06:36,2018-05-29
1372,a6b58794fd2ba533359a76c08df576e3,delivered,2018-05-14 15:18:23,2018-05-14 15:33:35,2018-05-14 13:46:00,2018-05-19 19:33:32,2018-06-08
1864,5792e0b1c8c8a2bf53af468c9a422c88,delivered,2018-07-26 13:25:14,2018-07-26 13:35:14,2018-07-26 12:42:00,2018-07-30 14:45:02,2018-08-09
2760,c3eb293fd154223498b6551a728203e8,delivered,2018-07-19 14:06:04,2018-07-19 14:22:51,2018-07-19 13:49:00,2018-07-24 19:35:36,2018-08-06
3473,b0c2a7d04b165525254254a728c50a4e,delivered,2018-06-07 13:28:30,2018-06-07 13:57:22,2018-06-07 13:22:00,2018-06-21 17:36:43,2018-07-04
3661,2033a4586b5bec3229ebc1675a8ae092,delivered,2018-06-12 10:10:25,2018-06-12 10:39:59,2018-06-12 10:09:00,2018-06-19 14:08:27,2018-07-19
4114,08adcddad19d3acf37d1fa01cb9ded1e,delivered,2018-06-27 11:16:44,2018-06-27 11:30:56,2018-06-27 10:57:00,2018-06-29 17:39:53,2018-07-18
4159,dee6298ce7d1fb2645141ef9972157aa,shipped,2018-04-30 14:06:12,2018-04-30 14:15:24,2018-04-30 12:59:00,NaT,2018-05-28


## 7. Delivery anomaly distribution

Delivery duration is calculated only where both purchase and customer-delivery timestamps exist. Extreme values are inspected rather than removed solely because they are large.

In [28]:
orders['delivery_days_audit'] = (
    orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']
).dt.total_seconds() / 86400

orders['delivery_days_audit'].describe(percentiles=[.01, .05, .5, .95, .99])

count    96476.000000
mean        12.558702
std          9.546530
min          0.533414
1%           1.825203
5%           3.015272
50%         10.217755
95%         29.276016
99%         46.049913
max        209.628611
Name: delivery_days_audit, dtype: float64

## 8. Order/payment/item reconciliation

Payment value and item value answer different questions. They should not be expected to match perfectly at the order level without considering freight, vouchers, installments, and the source's payment semantics.

The key requirement is that neither measure is accidentally multiplied by child-table joins.

In [29]:
order_item_value = items.groupby('order_id', as_index=False).agg(
    merchandise_value=('price', 'sum'),
    freight_value=('freight_value', 'sum'),
    item_count=('order_item_id', 'count')
)

order_payment_value = payments.groupby('order_id', as_index=False).agg(
    payment_value=('payment_value', 'sum'),
    payment_records=('payment_sequential', 'count')
)

reconciliation = order_item_value.merge(order_payment_value, on='order_id', how='outer', indicator=True)
reconciliation['_abs_difference'] = (reconciliation['merchandise_value'] + reconciliation['freight_value'] - reconciliation['payment_value']).abs()
reconciliation['_abs_difference'].describe(percentiles=[.5, .9, .95, .99])

count    9.866500e+04
mean     3.316222e-02
std      1.129109e+00
min      0.000000e+00
50%      0.000000e+00
90%      1.421085e-14
95%      2.842171e-14
99%      1.136868e-13
max      1.828100e+02
Name: _abs_difference, dtype: float64

In [34]:
payment_outliers = reconciliation[
    reconciliation["_abs_difference"] > 0.05
].sort_values("_abs_difference", ascending=False)

pd.DataFrame({
    "orders_checked": [len(reconciliation)],
    "difference_over_5_cents": [len(payment_outliers)],
    "difference_over_1_rupee": [
        (reconciliation["_abs_difference"] > 1).sum()
    ],
    "difference_over_10_rupees": [
        (reconciliation["_abs_difference"] > 10).sum()
    ],
    "max_difference": [reconciliation["_abs_difference"].max()]
})

,orders_checked,difference_over_5_cents,difference_over_1_rupee,difference_over_10_rupees,max_difference
0,99441,260,249,98,182.81


In [35]:
payment_outliers.head(20)

,order_id,merchandise_value,freight_value,item_count,payment_value,payment_records,_merge,_abs_difference
80176,ce6d150fb29ada17d2082f4847107665,1299.00,104.66,1.0,1586.47,1.0,both,182.81
42865,6e5fe7366a2e1bfbf3257dba0af1267f,179.19,108.72,6.0,406.92,1.0,both,119.01
43791,70b742795bc441e94a44a084b6d9ce7a,269.99,196.94,1.0,578.82,1.0,both,111.89
59228,996c7e73600ad3723e8627ab7bef81e4,559.90,28.00,1.0,664.43,1.0,both,76.53
43794,70b7e94ea46d3e8b5bc12a50186edaf0,167.88,45.27,3.0,274.84,1.0,both,61.69
73090,bc2c82b0ef78d2252b6176d1972db7c9,165.00,77.01,3.0,303.02,1.0,both,61.01
68015,af9ffff2ce6b3defd34fd4c78857a379,395.65,17.52,1.0,466.97,1.0,both,53.80
14745,262118ce178bb3e4590a3adcf6d62e6b,119.80,57.94,2.0,126.12,1.0,both,51.62
74503,bfdb5bbb06458d600a33d61f5f287472,297.00,51.93,1.0,394.36,1.0,both,45.43
54744,8d9c0dc8d5a2ce804f6b925d8f8e6c1d,209.80,44.65,2.0,293.89,1.0,both,39.44


## 9. Join-multiplication test

A deliberately broad join is shown only as a diagnostic. It must not be used as the universal KPI table.

In [30]:
master = (orders[['order_id', 'customer_id']]
          .merge(items[['order_id', 'product_id', 'price']], on='order_id', how='left')
          .merge(payments[['order_id', 'payment_value']], on='order_id', how='left')
          .merge(reviews[['order_id', 'review_id']], on='order_id', how='left'))

pd.Series({
    'source_orders': orders['order_id'].nunique(),
    'master_rows': len(master),
    'master_distinct_orders': master['order_id'].nunique(),
    'row_multiplier': round(len(master) / orders['order_id'].nunique(), 3)
})

source_orders              99441.000
master_rows               119143.000
master_distinct_orders     99441.000
row_multiplier                 1.198
dtype: float64

In [36]:
payment_reconciliation_detail = (
    reconciliation
    .merge(
        orders[["order_id", "order_status"]],
        on="order_id",
        how="left"
    )
    .merge(
        payments.groupby("order_id")["payment_type"]
        .apply(lambda x: ", ".join(sorted(x.dropna().unique())))
        .rename("payment_types"),
        on="order_id",
        how="left"
    )
)

payment_reconciliation_detail[
    payment_reconciliation_detail["_abs_difference"] > 10
][[
    "order_id",
    "order_status",
    "merchandise_value",
    "freight_value",
    "payment_value",
    "payment_types",
    "_abs_difference"
]].sort_values(
    "_abs_difference",
    ascending=False
).head(30)

,order_id,order_status,merchandise_value,freight_value,payment_value,payment_types,_abs_difference
80176,ce6d150fb29ada17d2082f4847107665,delivered,1299.00,104.66,1586.47,credit_card,182.81
42865,6e5fe7366a2e1bfbf3257dba0af1267f,delivered,179.19,108.72,406.92,credit_card,119.01
43791,70b742795bc441e94a44a084b6d9ce7a,delivered,269.99,196.94,578.82,credit_card,111.89
59228,996c7e73600ad3723e8627ab7bef81e4,delivered,559.90,28.00,664.43,credit_card,76.53
43794,70b7e94ea46d3e8b5bc12a50186edaf0,delivered,167.88,45.27,274.84,credit_card,61.69
73090,bc2c82b0ef78d2252b6176d1972db7c9,delivered,165.00,77.01,303.02,credit_card,61.01
68015,af9ffff2ce6b3defd34fd4c78857a379,delivered,395.65,17.52,466.97,credit_card,53.80
14745,262118ce178bb3e4590a3adcf6d62e6b,delivered,119.80,57.94,126.12,credit_card,51.62
74503,bfdb5bbb06458d600a33d61f5f287472,delivered,297.00,51.93,394.36,credit_card,45.43
54744,8d9c0dc8d5a2ce804f6b925d8f8e6c1d,delivered,209.80,44.65,293.89,credit_card,39.44


In [37]:
payment_reconciliation_detail[
    payment_reconciliation_detail["_abs_difference"] > 10
]["payment_types"].value_counts()

payment_types
credit_card    96
debit_card      2
Name: count, dtype: int64

In [38]:
card_reconciliation_detail = (
    payment_reconciliation_detail[
        payment_reconciliation_detail["_abs_difference"] > 10
    ]
    .merge(
        payments.groupby("order_id").agg(
            payment_record_count=("payment_sequential", "count"),
            payment_total=("payment_value", "sum"),
            payment_sequential_max=("payment_sequential", "max")
        ),
        on="order_id",
        how="left"
    )
)

card_reconciliation_detail[
    [
        "order_id",
        "order_status",
        "merchandise_value",
        "freight_value",
        "payment_value",
        "payment_types",
        "payment_record_count",
        "payment_sequential_max",
        "_abs_difference"
    ]
].sort_values(
    "_abs_difference",
    ascending=False
).head(30)

,order_id,order_status,merchandise_value,freight_value,payment_value,payment_types,payment_record_count,payment_sequential_max,_abs_difference
87,ce6d150fb29ada17d2082f4847107665,delivered,1299.00,104.66,1586.47,credit_card,1,1,182.81
46,6e5fe7366a2e1bfbf3257dba0af1267f,delivered,179.19,108.72,406.92,credit_card,1,1,119.01
48,70b742795bc441e94a44a084b6d9ce7a,delivered,269.99,196.94,578.82,credit_card,1,1,111.89
59,996c7e73600ad3723e8627ab7bef81e4,delivered,559.90,28.00,664.43,credit_card,1,1,76.53
49,70b7e94ea46d3e8b5bc12a50186edaf0,delivered,167.88,45.27,274.84,credit_card,1,1,61.69
80,bc2c82b0ef78d2252b6176d1972db7c9,delivered,165.00,77.01,303.02,credit_card,1,1,61.01
75,af9ffff2ce6b3defd34fd4c78857a379,delivered,395.65,17.52,466.97,credit_card,1,1,53.80
12,262118ce178bb3e4590a3adcf6d62e6b,delivered,119.80,57.94,126.12,credit_card,1,1,51.62
81,bfdb5bbb06458d600a33d61f5f287472,delivered,297.00,51.93,394.36,credit_card,1,1,45.43
54,8d9c0dc8d5a2ce804f6b925d8f8e6c1d,delivered,209.80,44.65,293.89,credit_card,1,1,39.44


In [39]:
card_reconciliation_detail[
    [
        "payment_record_count",
        "payment_sequential_max"
    ]
].value_counts().sort_index()

payment_record_count  payment_sequential_max
1                     1                         98
Name: count, dtype: int64

## Audit conclusion

Use the results above to classify findings before changing data. The corrected analytical model should use:

- order grain for orders, AOV, delivery, lateness and customer-level metrics
- item grain for category/product sales and freight
- payment grain for payment mix and payment values
- review grain or an explicitly aggregated order-review grain for review analysis

The audit is successful when KPI definitions are reproducible and no metric depends on accidental row multiplication.